# E11 House Prices：高维混合、缺失含义与集成

> Issue #9 / Epic #6 / 路线图 E11

## 你将学到
1. 缺失本身可能有业务含义（没有车库）
2. 目标常用 log1p
3. 树模型 + 线性模型简单加权集成

## 数据
- Kaggle House Prices `train.csv` → `data/house_prices_train.csv`
- 回退：OpenML `house_prices` 或合成 DEMO


In [ ]:
QUICK = True
RANDOM_STATE = 42

# 一键就绪：关警告 + 打印环境 + 设种子（勿再叠 JE.no_warning / filterwarnings）
from wanglaoshi import ready
ready(seed=RANDOM_STATE)


## 1. 加载


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

path = Path('../../data/house_prices_train.csv')
DEMO = False
if path.exists():
    train = pd.read_csv(path)
    print('local', train.shape)
else:
    try:
        from sklearn.datasets import fetch_openml
        raw = fetch_openml('house_prices', version=1, as_frame=True, parser='auto')
        train = raw.frame.copy()
        print('OpenML', train.shape)
    except Exception:
        DEMO = True
        print('合成 DEMO 房价数据')
        rng = np.random.RandomState(RANDOM_STATE)
        n = 800
        train = pd.DataFrame({
            'LotArea': rng.randint(1000, 20000, n),
            'OverallQual': rng.randint(1, 11, n),
            'YearBuilt': rng.randint(1900, 2011, n),
            'GarageArea': rng.choice([0, 200, 400, 600], n, p=[0.15, 0.3, 0.3, 0.25]),
            'Neighborhood': rng.choice(list('ABCDEFGH'), n),
            'SalePrice': None,
        })
        train['SalePrice'] = (
            5e4 + 30 * train['LotArea'] + 2e4 * train['OverallQual']
            + 50 * train['GarageArea'] + rng.normal(0, 2e4, n)
        ).clip(3e4, None)

if 'SalePrice' not in train.columns:
    for c in train.columns:
        if 'sale' in c.lower() or c.lower() == 'price':
            train = train.rename(columns={c: 'SalePrice'})
            break
print(train.head())
print('DEMO', DEMO)


## 2. EDA：缺失可能有含义


In [ ]:
from wanglaoshi import DataAnalyzer

eda_df = train.sample(min(len(train), 3000), random_state=RANDOM_STATE) if QUICK else train
analyzer = DataAnalyzer(eda_df)
miss = analyzer.missing_value_analysis()
display(miss.head(20))
# 示例：Garage* 同时缺失常表示无车库
garage_cols = [c for c in train.columns if c.startswith('Garage')]
print('garage cols', garage_cols[:8])


## 3. 简单预处理 + log1p 目标


In [ ]:
from sklearn.model_selection import train_test_split

work = train.copy()
y = np.log1p(work['SalePrice'].astype(float))
X = work.drop(columns=['SalePrice'])
# 仅演示：数值填中位数 + 类别频率编码（train-only）
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

num_cols = Xtr.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in Xtr.columns if c not in num_cols][:15]

med = Xtr[num_cols].median(numeric_only=True)
Xtr_n = Xtr[num_cols].fillna(med)
Xte_n = Xte[num_cols].fillna(med)
for c in cat_cols:
    freq = Xtr[c].astype(str).value_counts(normalize=True)
    Xtr_n[c + '_freq'] = Xtr[c].astype(str).map(freq).fillna(0)
    Xte_n[c + '_freq'] = Xte[c].astype(str).map(freq).fillna(0)
print(Xtr_n.shape)


## 4. QuickBaseline + 简单集成


In [ ]:
from wanglaoshi import CompetitionTools as CT
from sklearn.metrics import mean_squared_error, mean_absolute_error

baseline = CT.QuickBaseline()
results = baseline.run_all_models(Xtr_n, ytr, Xte_n, yte, task_type='regression')
comparison = baseline.compare_models(results, top_n=6)
display(comparison)

# 取前两名做等权平均（在 log 空间）
names = comparison['Model'].head(2).tolist()
preds = [np.asarray(results[n]['predictions']) for n in names]
blend = np.mean(preds, axis=0)
print('single best RMSE(log)', mean_squared_error(yte, preds[0], squared=False))
print('blend RMSE(log)', mean_squared_error(yte, blend, squared=False))
print('blend MAE(price)', mean_absolute_error(np.expm1(yte), np.expm1(blend)))


## 5. 模拟提交（还原到价格）


In [ ]:
from pathlib import Path
from wanglaoshi import CompetitionTools as CT

price_pred = np.expm1(blend)
price_true = np.expm1(yte.to_numpy())
out = Path('outputs/11_house_prices'); out.mkdir(parents=True, exist_ok=True)
sample = pd.DataFrame({'Id': np.arange(1, len(price_pred) + 1), 'SalePrice': price_true})
sample.to_csv(out / 'sample_submission.csv', index=False)
gen = CT.SubmissionGenerator()
sub = gen.create_submission(
    price_pred, str(out / 'sample_submission.csv'), str(out / 'submission.csv'), prediction_type='label'
)
print(gen.validate_submission(str(out / 'submission.csv'), str(out / 'sample_submission.csv')))
sub.head()


## 6. 检查清单
- [x] 缺失含义意识（Garage 等）
- [x] log1p 目标与还原提交
- [x] 简单加权集成
- [ ] 缺失指示列 + 类别缺失单独水平
- [ ] stacking 出折预测
